In [ ]:
import pandas as pd
from collections import Counter, defaultdict
import time

FILE = "tweets.csv"
CHUNK_SIZE = 100_000

print("Loading AmazonHelp conversation relationships...")

# ---------------------------------------------------------
# PASS 1
# Build a lookup of tweets that belong to AmazonHelp
# ---------------------------------------------------------

amazon_tweets = {}
amazon_support_ids = set()

start = time.time()

for i, chunk in enumerate(
    pd.read_csv(
        FILE,
        usecols=[
            "tweet_id",
            "author_id",
            "inbound",
            "text",
            "response_tweet_id",
            "in_response_to_tweet_id"
        ],
        chunksize=CHUNK_SIZE,
        low_memory=False
    )
):

    # Keep all AmazonHelp tweets
    amazon_rows = chunk[chunk["author_id"] == "AmazonHelp"]

    for _, row in amazon_rows.iterrows():
        tweet_id = str(row["tweet_id"])

        amazon_tweets[tweet_id] = {
            "author_id": row["author_id"],
            "inbound": row["inbound"],
            "text": row["text"],
            "response_tweet_id": row["response_tweet_id"],
            "in_response_to_tweet_id": row["in_response_to_tweet_id"]
        }

        # AmazonHelp outbound/support tweets
        if row["inbound"] == False:
            amazon_support_ids.add(tweet_id)

    if (i + 1) % 5 == 0:
        print(f"Processed {(i + 1) * CHUNK_SIZE:,} rows...")

print("\nAmazonHelp tweets loaded:", f"{len(amazon_tweets):,}")
print("AmazonHelp support tweets:", f"{len(amazon_support_ids):,}")
print("Time:", round(time.time() - start, 2), "seconds")


# ---------------------------------------------------------
# PASS 2
# Find customer tweets that AmazonHelp replied to
# ---------------------------------------------------------

customer_tweets = {}
customer_to_amazon = []

start = time.time()

for chunk in pd.read_csv(
    FILE,
    usecols=[
        "tweet_id",
        "author_id",
        "inbound",
        "text",
        "response_tweet_id",
        "in_response_to_tweet_id"
    ],
    chunksize=CHUNK_SIZE,
    low_memory=False
):

    inbound = chunk[chunk["inbound"] == True]

    for _, row in inbound.iterrows():
        tweet_id = str(row["tweet_id"])

        customer_tweets[tweet_id] = {
            "author_id": row["author_id"],
            "text": row["text"],
            "response_tweet_id": row["response_tweet_id"]
        }

        # Check whether this customer's tweet
        # is directly referenced by an AmazonHelp reply.
        response_ids = row["response_tweet_id"]

        if pd.notna(response_ids):
            response_list = str(response_ids).split()

            for response_id in response_list:
                if response_id in amazon_support_ids:
                    customer_to_amazon.append(
                        (tweet_id, response_id)
                    )

print("\nDirect customer -> AmazonHelp pairs:",
      f"{len(customer_to_amazon):,}")

print("Time:", round(time.time() - start, 2), "seconds")


# ---------------------------------------------------------
# Show examples
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("SAMPLE AMAZONHELP CUSTOMER -> SUPPORT CONVERSATIONS")
print("=" * 70)

shown = 0

for customer_id, support_id in customer_to_amazon:

    customer = customer_tweets.get(customer_id)
    support = amazon_tweets.get(support_id)

    if customer is None or support is None:
        continue

    print("\nCUSTOMER:")
    print(customer["text"])

    print("\nAMAZONHELP:")
    print(support["text"])

    print("\n" + "-" * 70)

    shown += 1

    if shown >= 20:
        break


# ---------------------------------------------------------
# Basic conversation statistics
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("AMAZONHELP BASIC STATISTICS")
print("=" * 70)

amazon_inbound = sum(
    1 for x in amazon_tweets.values()
    if x["inbound"] == True
)

amazon_outbound = sum(
    1 for x in amazon_tweets.values()
    if x["inbound"] == False
)

print("AmazonHelp inbound/customer tweets:",
      f"{amazon_inbound:,}")

print("AmazonHelp outbound/support tweets:",
      f"{amazon_outbound:,}")

print("Direct customer -> AmazonHelp pairs:",
      f"{len(customer_to_amazon):,}")

print("\nDone.")

Loading AmazonHelp conversation relationships...
Processed 500,000 rows...
Processed 1,000,000 rows...

AmazonHelp tweets loaded: 72,838
AmazonHelp support tweets: 72,838
Time: 11.67 seconds

Direct customer -> AmazonHelp pairs: 57,138
Time: 39.14 seconds

SAMPLE AMAZONHELP CUSTOMER -> SUPPORT CONVERSATIONS

CUSTOMER:
@AmazonHelp 電話で対応してもらいましたが改良されませんでした。
保証期間も過ぎてるので買い直しになるんでしょうね。

AMAZONHELP:
@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただきありがとうございました。ET

----------------------------------------------------------------------

CUSTOMER:
@AmazonHelp こちらこそありがとうございました。

AMAZONHELP:
@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお願いします。ET

----------------------------------------------------------------------

CUSTOMER:
amazonのfireTVstickが見れない😢

AMAZONHELP:
@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET

----------------------------------------------------------------------

CUSTOMER:
amazonプライムビデオ、再生エラーが多

In [ ]:
!ls

sample_data  tweets.csv


In [5]:
import pandas as pd
df=pd.read_csv('tweets.csv')
df.dim

AttributeError: 'DataFrame' object has no attribute 'dim'

In [ ]:
import pandas as pd
from collections import Counter, defaultdict
import os
import time

# ============================================================
# 1. FILE
# ============================================================

FILE = "tweets.csv"
CHUNK_SIZE = 100_000

print("File:", FILE)
print("Size:", round(os.path.getsize(FILE) / (1024**3), 2), "GB")

# ============================================================
# 2. READ ONLY THE HEADER
# ============================================================

columns = pd.read_csv(FILE, nrows=0).columns.tolist()

print("\nColumns:")
for col in columns:
    print(" -", col)

# ============================================================
# 3. BASIC DATASET ANALYSIS
# ============================================================

total_rows = 0
inbound_count = 0
outbound_count = 0

missing_counts = Counter()

# These counters help us identify the support/brand accounts.
author_counts = Counter()

start = time.time()

for i, chunk in enumerate(
    pd.read_csv(
        FILE,
        chunksize=CHUNK_SIZE,
        low_memory=False
    )
):

    total_rows += len(chunk)

    # Inbound = customer message
    inbound_count += (chunk["inbound"] == True).sum()

    # Outbound = support/brand message
    outbound_count += (chunk["inbound"] == False).sum()

    # Count authors
    author_counts.update(
        chunk["author_id"].dropna().astype(str)
    )

    # Missing values
    missing_counts.update(
        chunk.isna().sum().to_dict()
    )

    if (i + 1) % 5 == 0:
        print(
            f"Processed {(i + 1) * CHUNK_SIZE:,} rows..."
        )

elapsed = time.time() - start

# ============================================================
# 4. DATASET SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

print("Total rows:", f"{total_rows:,}")
print("Inbound/customer tweets:", f"{inbound_count:,}")
print("Outbound/support tweets:", f"{outbound_count:,}")
print("Processing time:", round(elapsed, 2), "seconds")

# ============================================================
# 5. TOP AUTHORS
# ============================================================

print("\n" + "=" * 60)
print("TOP 50 AUTHORS")
print("=" * 60)

for author, count in author_counts.most_common(50):
    print(f"{author:<35} {count:>10,}")

# ============================================================
# 6. MISSING VALUES
# ============================================================

print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

for col in columns:
    print(f"{col:<30} {missing_counts[col]:>12,}")

# ============================================================
# 7. IMPORTANT:
#    FIND AUTHORS THAT APPEAR ON OUTBOUND TWEETS
# ============================================================

print("\n" + "=" * 60)
print("OUTBOUND AUTHORS")
print("=" * 60)

outbound_authors = Counter()

for chunk in pd.read_csv(
    FILE,
    usecols=["author_id", "inbound"],
    chunksize=CHUNK_SIZE,
    low_memory=False
):

    outbound = chunk[chunk["inbound"] == False]

    outbound_authors.update(
        outbound["author_id"]
        .dropna()
        .astype(str)
    )

print("\nTop 50 accounts producing support replies:\n")

for author, count in outbound_authors.most_common(50):
    print(f"{author:<35} {count:>10,}")

print("\nDone.")

File: tweets.csv
Size: 0.17 GB

Columns:
 - tweet_id
 - author_id
 - inbound
 - created_at
 - text
 - response_tweet_id
 - in_response_to_tweet_id
Processed 500,000 rows...
Processed 1,000,000 rows...

DATASET SUMMARY
Total rows: 997,383
Inbound/customer tweets: 546,856
Outbound/support tweets: 450,527
Processing time: 5.45 seconds

TOP 50 AUTHORS
AmazonHelp                              75,266
AppleSupport                            31,363
Uber_Support                            20,398
Delta                                   15,571
SpotifyCares                            13,748
AmericanAir                             13,513
British_Airways                         10,645
Tesco                                   10,602
comcastcares                            10,368
TMobileHelp                             10,136
XboxSupport                              9,897
VirginTrains                             9,892
SouthwestAir                             9,620
hulu_support                           

In [ ]:
import pandas as pd
import numpy as np
import re
import time

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans

FILE = "tweets.csv"
CHUNK_SIZE = 100_000

print("Collecting AmazonHelp customer messages...")

# ---------------------------------------------------------
# PASS 1
# Get all AmazonHelp support tweet IDs
# ---------------------------------------------------------

amazon_support_ids = set()

for chunk in pd.read_csv(
    FILE,
    usecols=["tweet_id", "author_id", "inbound"],
    chunksize=CHUNK_SIZE,
    low_memory=False
):
    support = chunk[
        (chunk["author_id"] == "AmazonHelp") &
        (chunk["inbound"] == False)
    ]

    amazon_support_ids.update(
        support["tweet_id"].astype(str)
    )

print("AmazonHelp support tweets:",
      f"{len(amazon_support_ids):,}")


# ---------------------------------------------------------
# PASS 2
# Collect customer tweets that directly received
# an AmazonHelp response
# ---------------------------------------------------------

customer_messages = []

for chunk in pd.read_csv(
    FILE,
    usecols=[
        "tweet_id",
        "author_id",
        "inbound",
        "text",
        "response_tweet_id"
    ],
    chunksize=CHUNK_SIZE,
    low_memory=False
):

    inbound = chunk[chunk["inbound"] == True].copy()

    for _, row in inbound.iterrows():

        response_ids = row["response_tweet_id"]

        if pd.isna(response_ids):
            continue

        response_list = str(response_ids).split()

        # Keep only customers whose message
        # received an AmazonHelp response
        if any(rid in amazon_support_ids for rid in response_list):

            text = row["text"]

            if pd.notna(text) and str(text).strip():
                customer_messages.append(str(text))

print("\nUsable customer messages:",
      f"{len(customer_messages):,}")


# ---------------------------------------------------------
# Text cleaning
# ---------------------------------------------------------

def clean_text(text):

    text = str(text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove Twitter mentions
    text = re.sub(r"@\w+", " ", text)

    # Remove hashtags symbol but keep word
    text = re.sub(r"#", " ", text)

    # Remove common Twitter artifacts
    text = text.replace("&amp;", " ")
    text = text.replace("&gt;", " ")
    text = text.replace("&lt;", " ")

    # Keep letters/numbers
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    # Lowercase
    text = text.lower()

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


cleaned_messages = [
    clean_text(x)
    for x in customer_messages
]

# Remove extremely short messages
valid_pairs = [
    (original, cleaned)
    for original, cleaned in zip(
        customer_messages,
        cleaned_messages
    )
    if len(cleaned.split()) >= 3
]

original_messages = [x[0] for x in valid_pairs]
cleaned_messages = [x[1] for x in valid_pairs]

print("Messages after basic cleaning:",
      f"{len(cleaned_messages):,}")


# ---------------------------------------------------------
# TF-IDF
# ---------------------------------------------------------

print("\nBuilding TF-IDF representation...")

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=20,
    max_df=0.90,
    max_features=10_000
)

X = vectorizer.fit_transform(cleaned_messages)

print("TF-IDF shape:", X.shape)


# ---------------------------------------------------------
# Cluster messages
# ---------------------------------------------------------

N_CLUSTERS = 15

print("\nClustering messages into",
      N_CLUSTERS,
      "candidate groups...")

kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    random_state=42,
    batch_size=2048,
    n_init=5
)

labels = kmeans.fit_predict(X)


# ---------------------------------------------------------
# Display important terms for every cluster
# ---------------------------------------------------------

terms = vectorizer.get_feature_names_out()

print("\n" + "=" * 80)
print("CANDIDATE INTENT CLUSTERS")
print("=" * 80)

for cluster_id in range(N_CLUSTERS):

    indices = np.where(labels == cluster_id)[0]

    # Cluster center
    center = kmeans.cluster_centers_[cluster_id]

    top_indices = center.argsort()[-12:][::-1]

    top_terms = [
        terms[i]
        for i in top_indices
    ]

    print("\n" + "-" * 80)
    print(
        f"CLUSTER {cluster_id} "
        f"({len(indices):,} messages)"
    )

    print("Top terms:")
    print(", ".join(top_terms))

    print("\nRepresentative examples:")

    # Show first 5 examples
    for idx in indices[:5]:
        print(" -", original_messages[idx][:250])


print("\nDone.")

AmazonHelp support tweets: 76,269

Usable customer messages: 59,535
Messages after basic cleaning: 54,234

Building TF-IDF representation...
TF-IDF shape: (54234, 4534)

Clustering messages into 15 candidate groups...

CANDIDATE INTENT CLUSTERS

--------------------------------------------------------------------------------
CLUSTER 0 (25 messages)
Top terms:
coming, update, help, india, tomorrow, et que, et toujours, eta, etwas, eu, euch, euer

Representative examples:
 - @AmazonHelp Is there any update??
 - @AmazonHelp When u will be get back with update???
 - It's coming. Can't wait for @115833. 😍 https://t.co/q7cOPYN5AL
 - @AmazonHelp What's the update
 - @AmazonHelp What's the update

--------------------------------------------------------------------------------
CLUSTER 1 (1 messages)
Top terms:
dot, alexa, firetv, ordering, release, wtf, don know, ok, does, new, 10, know

Representative examples:
 - Ok I got an Alexa Dot as it was only $10 when ordering the new release firetv -

In [ ]:
import pandas as pd
import re
from collections import Counter

FILE = "tweets.csv"
CHUNK_SIZE = 100_000

# ---------------------------------------------------------
# Collect AmazonHelp support tweet IDs
# ---------------------------------------------------------

amazon_support_ids = set()

for chunk in pd.read_csv(
    FILE,
    usecols=["tweet_id", "author_id", "inbound"],
    chunksize=CHUNK_SIZE,
    low_memory=False
):
    support = chunk[
        (chunk["author_id"] == "AmazonHelp") &
        (chunk["inbound"] == False)
    ]

    amazon_support_ids.update(
        support["tweet_id"].astype(str)
    )

print("AmazonHelp support tweets:",
      f"{len(amazon_support_ids):,}")


# ---------------------------------------------------------
# Collect usable customer messages
# ---------------------------------------------------------

messages = []

for chunk in pd.read_csv(
    FILE,
    usecols=[
        "tweet_id",
        "inbound",
        "text",
        "response_tweet_id"
    ],
    chunksize=CHUNK_SIZE,
    low_memory=False
):

    inbound = chunk[chunk["inbound"] == True]

    for _, row in inbound.iterrows():

        if pd.isna(row["text"]) or pd.isna(row["response_tweet_id"]):
            continue

        response_ids = str(row["response_tweet_id"]).split()

        if any(
            rid in amazon_support_ids
            for rid in response_ids
        ):
            messages.append(str(row["text"]))


print("Usable messages:",
      f"{len(messages):,}")


# ---------------------------------------------------------
# Define broad semantic keyword groups
# ---------------------------------------------------------

categories = {
    "DELIVERY_ORDER": [
        "delivery", "delivered", "deliver", "package",
        "parcel", "shipping", "shipped", "shipment",
        "tracking", "track", "arrive", "arrived",
        "late", "delay", "delayed", "courier",
        "carrier", "order"
    ],

    "REFUND_PAYMENT": [
        "refund", "refunded", "money", "charge",
        "charged", "payment", "paid", "billing",
        "credit card", "debit card", "card",
        "fee", "price"
    ],

    "PRIME_MEMBERSHIP": [
        "prime", "membership", "subscription",
        "renew", "renewal", "trial", "cancel prime",
        "prime video"
    ],

    "ACCOUNT": [
        "account", "password", "login", "log in",
        "sign in", "email", "verification",
        "code", "security", "locked"
    ],

    "PRODUCT_DEVICE": [
        "echo", "alexa", "kindle", "fire tv",
        "firetv", "tablet", "device", "app",
        "application", "tv", "music"
    ],

    "ORDER_PRODUCT": [
        "product", "item", "replacement", "replace",
        "broken", "damaged", "missing", "wrong item",
        "incorrect", "stock", "available"
    ],

    "CUSTOMER_SERVICE": [
        "customer service", "support", "help",
        "agent", "representative", "call",
        "phone", "contact", "manager"
    ]
}


# ---------------------------------------------------------
# Count messages matching each broad category
# ---------------------------------------------------------

category_counts = Counter()

for message in messages:

    text = message.lower()

    for category, keywords in categories.items():

        if any(keyword in text for keyword in keywords):
            category_counts[category] += 1


print("\n" + "=" * 70)
print("BROAD CATEGORY COVERAGE")
print("=" * 70)

for category, count in category_counts.most_common():

    percentage = 100 * count / len(messages)

    print(
        f"{category:<22} "
        f"{count:>8,} "
        f"({percentage:>5.1f}%)"
    )


# ---------------------------------------------------------
# Show examples for each category
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("EXAMPLES BY BROAD CATEGORY")
print("=" * 70)

for category, keywords in categories.items():

    print("\n" + "-" * 70)
    print(category)

    shown = 0

    for message in messages:

        text = message.lower()

        if any(keyword in text for keyword in keywords):

            print(" -", message[:300])

            shown += 1

            if shown >= 10:
                break


print("\nDone.")

AmazonHelp support tweets: 80,552


ParserError: Error tokenizing data. C error: EOF inside string starting at row 1123666

In [ ]:
import pandas as pd
import re
from collections import Counter, defaultdict

FILE = "tweets.csv"
CHUNK_SIZE = 100_000

# ---------------------------------------------------------
# PASS 1
# Store all AmazonHelp support tweets
# ---------------------------------------------------------

amazon_support = {}

for chunk in pd.read_csv(
    FILE,
    usecols=[
        "tweet_id",
        "author_id",
        "inbound",
        "text",
        "in_response_to_tweet_id"
    ],
    chunksize=CHUNK_SIZE,
    low_memory=False
):

    support = chunk[
        (chunk["author_id"] == "AmazonHelp") &
        (chunk["inbound"] == False)
    ]

    for _, row in support.iterrows():

        amazon_support[str(row["tweet_id"])] = {
            "text": row["text"],
            "parent": (
                str(row["in_response_to_tweet_id"])
                if pd.notna(row["in_response_to_tweet_id"])
                else None
            )
        }

print(
    "AmazonHelp support tweets:",
    f"{len(amazon_support):,}"
)


# ---------------------------------------------------------
# PASS 2
# Collect customer -> AmazonHelp pairs
# ---------------------------------------------------------

pairs = []

for chunk in pd.read_csv(
    FILE,
    usecols=[
        "tweet_id",
        "inbound",
        "text",
        "response_tweet_id"
    ],
    chunksize=CHUNK_SIZE,
    low_memory=False
):

    inbound = chunk[chunk["inbound"] == True]

    for _, row in inbound.iterrows():

        if pd.isna(row["text"]) or pd.isna(row["response_tweet_id"]):
            continue

        response_ids = str(row["response_tweet_id"]).split()

        for response_id in response_ids:

            if response_id in amazon_support:

                pairs.append({
                    "customer": str(row["text"]),
                    "reply": str(
                        amazon_support[response_id]["text"]
                    )
                })


print(
    "Customer -> AmazonHelp pairs:",
    f"{len(pairs):,}"
)


# ---------------------------------------------------------
# Broad intent keywords
# ---------------------------------------------------------

intent_keywords = {

    "DELIVERY_DELAY": [
        "late", "delayed", "delay",
        "not arrived", "hasn't arrived",
        "haven't received", "still waiting",
        "where is my order", "where is my package",
        "delivery date", "delivery today"
    ],

    "DELIVERED_NOT_RECEIVED": [
        "marked delivered",
        "says delivered",
        "shows delivered",
        "tracking says delivered",
        "not received",
        "didn't receive",
        "did not receive",
        "never received"
    ],

    "SHIPPING_TRACKING": [
        "tracking", "track",
        "shipping", "shipped",
        "dispatch", "carrier",
        "courier"
    ],

    "REFUND": [
        "refund", "refunded",
        "money back",
        "give my money",
        "get my money back"
    ],

    "PAYMENT_CHARGE": [
        "charged", "charge",
        "payment", "billing",
        "credit card", "debit card",
        "card charged", "false charge"
    ],

    "PRIME": [
        "amazon prime",
        "prime membership",
        "prime trial",
        "prime subscription",
        "cancel prime",
        "prime member"
    ],

    "ACCOUNT_ACCESS": [
        "account",
        "password",
        "login",
        "log in",
        "sign in",
        "verification",
        "verification code",
        "can't access",
        "cannot access"
    ],

    "PRODUCT_DEVICE": [
        "echo",
        "alexa",
        "kindle",
        "fire tv",
        "firetv",
        "tablet",
        "device",
        "app",
        "prime video"
    ],

    "PRODUCT_PROBLEM": [
        "damaged",
        "broken",
        "faulty",
        "defective",
        "wrong item",
        "missing item",
        "counterfeit",
        "fake item"
    ],

    "CUSTOMER_SERVICE": [
        "customer service",
        "support",
        "agent",
        "representative",
        "call",
        "phone",
        "contact",
        "manager"
    ]
}


# ---------------------------------------------------------
# Classify examples using simple keyword priority
# ---------------------------------------------------------
# This is NOT our final classifier.
# It is only being used to inspect historical replies.

def detect_intent(text):

    text = text.lower()

    # More specific categories first

    priority = [
        "DELIVERED_NOT_RECEIVED",
        "DELIVERY_DELAY",
        "REFUND",
        "PAYMENT_CHARGE",
        "PRIME",
        "ACCOUNT_ACCESS",
        "PRODUCT_DEVICE",
        "PRODUCT_PROBLEM",
        "SHIPPING_TRACKING",
        "CUSTOMER_SERVICE"
    ]

    for intent in priority:

        for keyword in intent_keywords[intent]:

            if keyword in text:
                return intent

    return "OTHER"


# ---------------------------------------------------------
# Group historical replies
# ---------------------------------------------------------

reply_groups = defaultdict(list)

for pair in pairs:

    intent = detect_intent(pair["customer"])

    if len(reply_groups[intent]) < 500:
        reply_groups[intent].append(pair)


# ---------------------------------------------------------
# Print statistics
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("CUSTOMER INTENT COVERAGE")
print("=" * 80)

counts = Counter()

for pair in pairs:
    counts[detect_intent(pair["customer"])] += 1

for intent, count in counts.most_common():

    percentage = 100 * count / len(pairs)

    print(
        f"{intent:<25}"
        f"{count:>8,}"
        f" ({percentage:>5.1f}%)"
    )


# ---------------------------------------------------------
# Show historical resolution examples
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("HISTORICAL RESOLUTION EXAMPLES")
print("=" * 80)

for intent in [
    "DELIVERY_DELAY",
    "DELIVERED_NOT_RECEIVED",
    "SHIPPING_TRACKING",
    "REFUND",
    "PAYMENT_CHARGE",
    "PRIME",
    "ACCOUNT_ACCESS",
    "PRODUCT_DEVICE",
    "PRODUCT_PROBLEM",
    "CUSTOMER_SERVICE"
]:

    examples = reply_groups.get(intent, [])

    print("\n" + "-" * 80)
    print(intent)

    if not examples:
        print("No examples found.")
        continue

    for pair in examples[:8]:

        print("\nCUSTOMER:")
        print(pair["customer"][:350])

        print("\nAMAZONHELP:")
        print(pair["reply"][:500])

        print("\n" + "." * 60)


print("\nDone.")

AmazonHelp support tweets: 1,246
Customer -> AmazonHelp pairs: 843

CUSTOMER INTENT COVERAGE
OTHER                         487 ( 57.8%)
PRODUCT_DEVICE                 78 (  9.3%)
CUSTOMER_SERVICE               78 (  9.3%)
SHIPPING_TRACKING              42 (  5.0%)
DELIVERY_DELAY                 42 (  5.0%)
ACCOUNT_ACCESS                 36 (  4.3%)
REFUND                         29 (  3.4%)
PRIME                          18 (  2.1%)
PAYMENT_CHARGE                 14 (  1.7%)
PRODUCT_PROBLEM                10 (  1.2%)
DELIVERED_NOT_RECEIVED          9 (  1.1%)

HISTORICAL RESOLUTION EXAMPLES

--------------------------------------------------------------------------------
DELIVERY_DELAY

CUSTOMER:
@115821 @AmazonHelp why is my order at my local courier for the last 6 days and still hasn’t been delivered to me?? Over 1 week late 😡

AMAZONHELP:
@115831 I'm sorry for the wait. Please reach out to us so we can take a closer look at this delivery: https://t.co/JzP7hlA23B ^SH

...............

In [ ]:
# Check what one pair looks like

print(type(pairs))
print("Number of pairs:", len(pairs))

print("\nFirst pair:")
print(pairs[0])

print("\nType of first pair:")
print(type(pairs[0]))

<class 'list'>
Number of pairs: 843

First pair:
{'customer': '@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎてるので買い直しになるんでしょうね。', 'reply': '@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただきありがとうございました。ET'}

Type of first pair:
<class 'dict'>


In [ ]:
# STEP 6: Analyze the OTHER messages

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans

# Convert the existing list of dictionaries into a DataFrame
pairs_df = pd.DataFrame(pairs)

print("Total pairs:", len(pairs_df))
print("Columns:", pairs_df.columns.tolist())


# ---------------------------------------------------------
# Reuse the same heuristic intent function from Step 5
# ---------------------------------------------------------

def detect_intent(text):
    text = str(text).lower()

    if any(x in text for x in [
        "delivered but", "marked delivered", "says delivered",
        "shows delivered", "delivery says delivered"
    ]):
        return "DELIVERED_NOT_RECEIVED"

    if any(x in text for x in [
        "late", "delayed", "delay", "hasn't arrived",
        "haven't received", "not arrived", "where is my order"
    ]):
        return "DELIVERY_DELAY"

    if any(x in text for x in [
        "tracking", "track my", "tracking number",
        "shipping information", "shipping date", "shipped"
    ]):
        return "SHIPPING_TRACKING"

    if any(x in text for x in [
        "refund", "money back", "refunded"
    ]):
        return "REFUND"

    if any(x in text for x in [
        "charged", "charge", "payment", "billing",
        "charged me", "amazon payments"
    ]):
        return "PAYMENT_CHARGE"

    if any(x in text for x in [
        "prime", "prime membership", "prime trial",
        "prime video"
    ]):
        return "PRIME"

    if any(x in text for x in [
        "login", "log in", "password", "account",
        "locked", "can't access", "cannot access"
    ]):
        return "ACCOUNT_ACCESS"

    if any(x in text for x in [
        "fire tv", "firetv", "kindle", "echo",
        "alexa", "device", "app"
    ]):
        return "PRODUCT_DEVICE"

    if any(x in text for x in [
        "broken", "damaged", "faulty", "doesn't work",
        "not working", "counterfeit", "fake item"
    ]):
        return "PRODUCT_PROBLEM"

    if any(x in text for x in [
        "customer service", "support", "help",
        "contact", "call"
    ]):
        return "CUSTOMER_SERVICE"

    return "OTHER"


# Apply the heuristic classifier to every customer message
pairs_df["heuristic_intent"] = pairs_df["customer"].apply(detect_intent)

print("\nIntent distribution:")
print(
    pairs_df["heuristic_intent"]
    .value_counts()
    .to_string()
)


# ---------------------------------------------------------
# Select OTHER
# ---------------------------------------------------------

other = pairs_df[
    pairs_df["heuristic_intent"] == "OTHER"
].copy()

print("\nOTHER messages:", len(other))


# ---------------------------------------------------------
# Clean customer text
# ---------------------------------------------------------

other["customer_clean"] = (
    other["customer"]
    .fillna("")
    .astype(str)
    .str.replace(r"http\S+", " ", regex=True)
    .str.replace(r"@\w+", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

other = other[
    other["customer_clean"].str.len() >= 10
].copy()

print("Usable OTHER messages:", len(other))


# ---------------------------------------------------------
# TF-IDF
# ---------------------------------------------------------

vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5
)

X = vectorizer.fit_transform(other["customer_clean"])

print("TF-IDF shape:", X.shape)


# ---------------------------------------------------------
# Clustering
# ---------------------------------------------------------

N_CLUSTERS = 15

kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    random_state=42,
    batch_size=2048,
    n_init=5
)

labels = kmeans.fit_predict(X)

other["cluster"] = labels


# ---------------------------------------------------------
# Important terms per cluster
# ---------------------------------------------------------

terms = np.array(
    vectorizer.get_feature_names_out()
)

print("\n" + "=" * 80)
print("OTHER MESSAGE CLUSTERS")
print("=" * 80)

for cluster_id in range(N_CLUSTERS):

    mask = other["cluster"] == cluster_id
    count = mask.sum()

    center = kmeans.cluster_centers_[cluster_id]

    top_indices = center.argsort()[-15:][::-1]
    top_terms = terms[top_indices]

    print(
        f"\nCluster {cluster_id:2d} | "
        f"{count:6d} messages "
        f"({count / len(other) * 100:5.1f}%)"
    )

    print("Terms:", ", ".join(top_terms))


# ---------------------------------------------------------
# Representative examples
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("REPRESENTATIVE EXAMPLES")
print("=" * 80)

for cluster_id in range(N_CLUSTERS):

    cluster_mask = other["cluster"] == cluster_id
    cluster_indices = np.where(cluster_mask)[0]

    if len(cluster_indices) == 0:
        continue

    cluster_X = X[cluster_indices]

    # Distance from each message to cluster centroid
    distances = np.linalg.norm(
        cluster_X.toarray()
        - kmeans.cluster_centers_[cluster_id],
        axis=1
    )

    closest = cluster_indices[
        np.argsort(distances)[:5]
    ]

    print("\n" + "-" * 80)
    print(f"CLUSTER {cluster_id}")
    print("-" * 80)

    for idx in closest:

        row = other.iloc[idx]

        print("\nCUSTOMER:")
        print(row["customer"])

        print("\nAMAZONHELP:")
        print(row["reply"])

Total pairs: 843
Columns: ['customer', 'reply']

Intent distribution:
heuristic_intent
CUSTOMER_SERVICE          410
OTHER                     157
PRODUCT_DEVICE             68
PRIME                      62
ACCOUNT_ACCESS             36
DELIVERY_DELAY             35
REFUND                     27
SHIPPING_TRACKING          16
PAYMENT_CHARGE             14
PRODUCT_PROBLEM            13
DELIVERED_NOT_RECEIVED      5

OTHER messages: 157
Usable OTHER messages: 154
TF-IDF shape: (154, 15)

OTHER MESSAGE CLUSTERS

Cluster  0 |      8 messages (  5.2%)
Terms: delivery, amazon, today, ordered, delivered, shipping, thanks, package, que, service, order, don, hey, day, amp

Cluster  1 |     82 messages ( 53.2%)
Terms: hey, thanks, today, service, que, package, shipping, ordered, order, don, delivery, delivered, day, amp, amazon

Cluster  2 |      9 messages (  5.8%)
Terms: amazon, thanks, today, service, que, package, shipping, ordered, order, don, hey, delivery, delivered, day, amp

Cluster  3 |

In [ ]:
# STEP 7: Analyze historical resolution behavior

import pandas as pd
import re

print("Total pairs:", len(pairs_df))


# ---------------------------------------------------------
# Assign broader resolution/action categories
# ---------------------------------------------------------

def detect_resolution(reply):

    text = str(reply).lower()

    # Human / direct support
    if any(x in text for x in [
        "phone or chat",
        "contact us",
        "reach us",
        "call or chat",
        "reach out",
        "live troubleshooting",
        "real time",
        "secure link",
        "provide your details",
        "fill in the form"
    ]):
        return "HUMAN_SUPPORT"

    # Tracking / order status
    if any(x in text for x in [
        "tracking",
        "order status",
        "delivery date",
        "shipping date",
        "estimated delivery",
        "order details"
    ]):
        return "TRACKING_STATUS"

    # Refund / return / replacement
    if any(x in text for x in [
        "refund",
        "replacement",
        "return",
        "return options"
    ]):
        return "REFUND_RETURN"

    # Troubleshooting / technical
    if any(x in text for x in [
        "troubleshooting",
        "restart",
        "reinstall",
        "power cycle",
        "help pages",
        "error message",
        "steps",
        "troubleshoot"
    ]):
        return "TROUBLESHOOTING"

    # Policy / informational answer
    if any(x in text for x in [
        "you can",
        "you will",
        "information",
        "for details",
        "instructions",
        "here's more info",
        "please check"
    ]):
        return "INFORMATION"

    # Asking for more information
    if any(x in text for x in [
        "can you tell",
        "could you tell",
        "please provide",
        "what does",
        "what is",
        "how long",
        "which",
        "are you"
    ]):
        return "MORE_INFORMATION"

    # Generic acknowledgement
    if any(x in text for x in [
        "thanks",
        "thank you",
        "sorry",
        "glad",
        "hope this helps"
    ]):
        return "ACKNOWLEDGEMENT"

    return "OTHER"


pairs_df["resolution_type"] = pairs_df["reply"].apply(
    detect_resolution
)


# ---------------------------------------------------------
# Overall resolution distribution
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("OVERALL HISTORICAL RESOLUTION TYPES")
print("=" * 80)

resolution_counts = pairs_df["resolution_type"].value_counts()

for name, count in resolution_counts.items():
    print(
        f"{name:20s} "
        f"{count:7d} "
        f"({count / len(pairs_df) * 100:5.1f}%)"
    )


# ---------------------------------------------------------
# Resolution type by heuristic intent
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("RESOLUTION TYPE BY CUSTOMER INTENT")
print("=" * 80)

cross_tab = pd.crosstab(
    pairs_df["heuristic_intent"],
    pairs_df["resolution_type"],
    normalize="index"
) * 100

print(
    cross_tab.round(1).to_string()
)


# ---------------------------------------------------------
# Representative examples for every resolution type
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("REPRESENTATIVE RESOLUTION EXAMPLES")
print("=" * 80)

for resolution in resolution_counts.index:

    subset = pairs_df[
        pairs_df["resolution_type"] == resolution
    ]

    print("\n" + "-" * 80)
    print(resolution)
    print("-" * 80)

    for _, row in subset.head(8).iterrows():

        print("\nCUSTOMER:")
        print(row["customer"])

        print("\nAMAZONHELP:")
        print(row["reply"])


Total pairs: 843

OVERALL HISTORICAL RESOLUTION TYPES
OTHER                    376 ( 44.6%)
ACKNOWLEDGEMENT          133 ( 15.8%)
HUMAN_SUPPORT            120 ( 14.2%)
INFORMATION               75 (  8.9%)
MORE_INFORMATION          55 (  6.5%)
TRACKING_STATUS           48 (  5.7%)
REFUND_RETURN             22 (  2.6%)
TROUBLESHOOTING           14 (  1.7%)

RESOLUTION TYPE BY CUSTOMER INTENT
resolution_type         ACKNOWLEDGEMENT  HUMAN_SUPPORT  INFORMATION  MORE_INFORMATION  OTHER  REFUND_RETURN  TRACKING_STATUS  TROUBLESHOOTING
heuristic_intent                                                                                                                             
ACCOUNT_ACCESS                     19.4           22.2         22.2               5.6   19.4            2.8              0.0              8.3
CUSTOMER_SERVICE                   15.6           13.7          7.3               7.3   49.3            1.2              5.4              0.2
DELIVERED_NOT_RECEIVED             40.

In [ ]:
# STEP 8: Inspect candidate final intents

import pandas as pd

candidate_intents = [
    "DELIVERY_DELAY",
    "DELIVERED_NOT_RECEIVED",
    "SHIPPING_TRACKING",
    "REFUND",
    "PAYMENT_CHARGE",
    "PRIME",
    "ACCOUNT_ACCESS",
    "PRODUCT_DEVICE",
    "PRODUCT_PROBLEM",
    "GENERAL_SUPPORT",
    "OTHER"
]

print("=" * 80)
print("CANDIDATE INTENT SAMPLE REVIEW")
print("=" * 80)

for intent in candidate_intents:

    print("\n" + "#" * 80)
    print(intent)
    print("#" * 80)

    # Use the existing heuristic labels as a temporary sampling mechanism.
    # These are NOT ground-truth labels.
    subset = pairs_df[
        pairs_df["heuristic_intent"] == intent
    ]

    print("Available examples:", len(subset))

    if len(subset) == 0:
        continue

    # Fixed random sample so the result can be reproduced.
    sample = subset.sample(
        n=min(10, len(subset)),
        random_state=42
    )

    for i, (_, row) in enumerate(sample.iterrows(), 1):

        print(f"\nExample {i}")

        print("CUSTOMER:")
        print(str(row["customer"])[:1000])

        print("\nAMAZONHELP:")
        print(str(row["reply"])[:1000])

CANDIDATE INTENT SAMPLE REVIEW

################################################################################
DELIVERY_DELAY
################################################################################
Available examples: 35

Example 1
CUSTOMER:
@AmazonHelp I’ve had one saying it will be late but no reason why

AMAZONHELP:
@120262 Sorry to hear that. We'll be happy to look into that for you. Please reach out to us here: https://t.co/JzP7hlA23B ^RS

Example 2
CUSTOMER:
@AmazonHelp Not in the mailbox, not on the porch, not in my house, not at the neighbors. This was a Prime order, late/lost again.

AMAZONHELP:
@118235 I'm sorry those steps didn't help! Who was the carrier for this order? You can check here: https://t.co/Y5jpI9gRhE ^BL

Example 3
CUSTOMER:
@AmazonHelp All m getting is apologies but no refund.. everything is online, unable to understand the delay!!

AMAZONHELP:
@119700 You must have received a correspondence to your registered email address, request you to check the

In [ ]:
import pandas as pd

df = pd.read_csv("/content/tweets.csv")

print("Data loaded successfully")
print("Shape:", df.shape)

Data loaded successfully
Shape: (18317, 7)


In [ ]:
# STEP 9: Inspect conversation structure correctly

import pandas as pd

print("=" * 80)
print("CONVERSATION STRUCTURE")
print("=" * 80)

print("Original dataframe shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(
    df[
        [
            "tweet_id",
            "author_id",
            "inbound",
            "response_tweet_id",
            "in_response_to_tweet_id"
        ]
    ].isnull().sum()
)

print("\nUnique authors:", df["author_id"].nunique())
print("Unique tweets:", df["tweet_id"].nunique())


# ---------------------------------------------------------
# AmazonHelp tweets
# ---------------------------------------------------------

amazon_rows = df[
    df["author_id"].astype(str) == "AmazonHelp"
].copy()

print("\nAmazonHelp tweets:", len(amazon_rows))


# ---------------------------------------------------------
# Find customer tweets that AmazonHelp directly replied to
#
# AmazonHelp tweet:
#     in_response_to_tweet_id = customer's tweet_id
# ---------------------------------------------------------

customer_tweet_ids = amazon_rows[
    amazon_rows["in_response_to_tweet_id"].notna()
]["in_response_to_tweet_id"]

customer_to_amazon = df[
    df["tweet_id"].isin(customer_tweet_ids)
].copy()

print(
    "Customer tweets with direct AmazonHelp response:",
    len(customer_to_amazon)
)


# ---------------------------------------------------------
# Customer interaction frequency
# ---------------------------------------------------------

customer_counts = (
    customer_to_amazon["author_id"]
    .value_counts()
)

print("\nCustomer interaction statistics:")

if len(customer_counts) > 0:
    print(customer_counts.describe())

    print(
        "\nCustomers with more than 1 interaction:",
        (customer_counts > 1).sum()
    )

    print(
        "Customers with more than 5 interactions:",
        (customer_counts > 5).sum()
    )

    print(
        "Customers with more than 10 interactions:",
        (customer_counts > 10).sum()
    )
else:
    print("No direct customer-AmazonHelp interactions found.")


# ---------------------------------------------------------
# Inspect actual conversation relationships
# ---------------------------------------------------------

print("\nSample conversation relationships:")

sample = amazon_rows[
    amazon_rows["in_response_to_tweet_id"].notna()
][
    [
        "tweet_id",
        "author_id",
        "text",
        "response_tweet_id",
        "in_response_to_tweet_id"
    ]
].head(10)

print(sample.to_string(index=False))

CONVERSATION STRUCTURE
Original dataframe shape: (18317, 7)

Columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']

Missing values:
tweet_id                      0
author_id                     0
inbound                       0
response_tweet_id          6010
in_response_to_tweet_id    4766
dtype: int64

Unique authors: 5197
Unique tweets: 18317

AmazonHelp tweets: 1246
Customer tweets with direct AmazonHelp response: 1073

Customer interaction statistics:
count    480.000000
mean       2.235417
std        2.249750
min        1.000000
25%        1.000000
50%        1.000000
75%        3.000000
max       18.000000
Name: count, dtype: float64

Customers with more than 1 interaction: 234
Customers with more than 5 interactions: 26
Customers with more than 10 interactions: 10

Sample conversation relationships:
 tweet_id  author_id                                                                                                   

In [ ]:
# STEP 10: Analyze AmazonHelp conversation threads

import pandas as pd
from collections import defaultdict, deque

print("=" * 80)
print("AMAZONHELP CONVERSATION THREAD ANALYSIS")
print("=" * 80)


# ---------------------------------------------------------
# Keep only tweets connected to AmazonHelp interactions
# ---------------------------------------------------------

amazon_ids = set(
    amazon_rows["tweet_id"].astype(str)
)

amazon_parent_ids = set(
    amazon_rows[
        amazon_rows["in_response_to_tweet_id"].notna()
    ]["in_response_to_tweet_id"]
    .astype(int)
    .astype(str)
)

amazon_child_ids = set()

for value in amazon_rows["response_tweet_id"].dropna():
    for tweet_id in str(value).split(","):
        amazon_child_ids.add(tweet_id.strip())


# All tweets directly connected to AmazonHelp
connected_ids = (
    amazon_ids |
    amazon_parent_ids |
    amazon_child_ids
)

connected = df[
    df["tweet_id"].astype(str).isin(connected_ids)
].copy()

print("Tweets connected to AmazonHelp:", len(connected))

print(
    "Unique authors in connected tweets:",
    connected["author_id"].nunique()
)


# ---------------------------------------------------------
# Build parent -> children relationships
# ---------------------------------------------------------

parent_map = {}

for _, row in connected.iterrows():

    tweet_id = str(row["tweet_id"])

    parent = row["in_response_to_tweet_id"]

    if pd.notna(parent):
        parent_map[tweet_id] = str(int(parent))


# ---------------------------------------------------------
# Find root tweet for each connected tweet
# ---------------------------------------------------------

def find_root(tweet_id):

    visited = set()
    current = tweet_id

    while current in parent_map:

        if current in visited:
            break

        visited.add(current)

        current = parent_map[current]

    return current


connected["conversation_root"] = (
    connected["tweet_id"]
    .astype(str)
    .apply(find_root)
)


# ---------------------------------------------------------
# Conversation statistics
# ---------------------------------------------------------

conversation_sizes = (
    connected
    .groupby("conversation_root")
    .size()
)

print("\nConversation statistics:")

print(
    conversation_sizes.describe()
)

print(
    "\nNumber of conversation threads:",
    len(conversation_sizes)
)

print(
    "Threads with more than 2 tweets:",
    (conversation_sizes > 2).sum()
)

print(
    "Threads with more than 5 tweets:",
    (conversation_sizes > 5).sum()
)

print(
    "Largest conversation thread:",
    conversation_sizes.max()
)


# ---------------------------------------------------------
# Show largest conversations
# ---------------------------------------------------------

print("\nLargest conversation threads:")

print(
    conversation_sizes
    .sort_values(ascending=False)
    .head(10)
)


# ---------------------------------------------------------
# Inspect one larger conversation
# ---------------------------------------------------------

largest_root = conversation_sizes.idxmax()

largest_conversation = connected[
    connected["conversation_root"] == largest_root
].sort_values("tweet_id")

print("\nExample largest conversation:")

print(
    largest_conversation[
        [
            "tweet_id",
            "author_id",
            "inbound",
            "text",
            "in_response_to_tweet_id"
        ]
    ].to_string(index=False)
)

AMAZONHELP CONVERSATION THREAD ANALYSIS
Tweets connected to AmazonHelp: 2555
Unique authors in connected tweets: 486

Conversation statistics:
count    497.000000
mean       5.140845
std        5.376573
min        2.000000
25%        2.000000
50%        3.000000
75%        6.000000
max       45.000000
dtype: float64

Number of conversation threads: 497
Threads with more than 2 tweets: 317
Threads with more than 5 tweets: 138
Largest conversation thread: 45

Largest conversation threads:
conversation_root
21586    45
16882    43
13165    35
18941    35
17216    31
21045    30
16824    28
12548    28
13897    27
20468    25
dtype: int64

Example largest conversation:
 tweet_id  author_id  inbound                                                                                                                                                                                                                                                                                           text  in_respo

In [ ]:
# STEP 11: Validate customer/thread relationships

print("=" * 80)
print("CUSTOMER AND CONVERSATION OVERLAP")
print("=" * 80)


# ---------------------------------------------------------
# Number of threads per customer
# ---------------------------------------------------------

customer_thread_counts = (
    connected
    .groupby("author_id")["conversation_root"]
    .nunique()
)

print("\nThreads per customer:")
print(customer_thread_counts.describe())

print(
    "\nCustomers appearing in more than 1 thread:",
    (customer_thread_counts > 1).sum()
)

print(
    "Customers appearing in more than 5 threads:",
    (customer_thread_counts > 5).sum()
)

print(
    "Customers appearing in more than 10 threads:",
    (customer_thread_counts > 10).sum()
)


# ---------------------------------------------------------
# Distribution of customer tweets inside threads
# ---------------------------------------------------------

customer_thread_pairs = (
    connected[
        connected["inbound"] == True
    ]
    .groupby("conversation_root")
    .size()
)

print("\nCustomer-tweet count per conversation:")
print(customer_thread_pairs.describe())

print(
    "\nThreads with no customer tweets:",
    (customer_thread_pairs == 0).sum()
)

print(
    "Threads with more than 1 customer tweet:",
    (customer_thread_pairs > 1).sum()
)

print(
    "Threads with more than 5 customer tweets:",
    (customer_thread_pairs > 5).sum()
)


# ---------------------------------------------------------
# AmazonHelp responses per conversation
# ---------------------------------------------------------

support_thread_pairs = (
    connected[
        connected["author_id"].astype(str) == "AmazonHelp"
    ]
    .groupby("conversation_root")
    .size()
)

print("\nAmazonHelp tweet count per conversation:")
print(support_thread_pairs.describe())

print(
    "\nThreads with more than 1 AmazonHelp reply:",
    (support_thread_pairs > 1).sum()
)

print(
    "Threads with more than 5 AmazonHelp replies:",
    (support_thread_pairs > 5).sum()
)


# ---------------------------------------------------------
# Show customers with many threads
# ---------------------------------------------------------

print("\nCustomers appearing in the most detected threads:")

print(
    customer_thread_counts
    .sort_values(ascending=False)
    .head(10)
)

CUSTOMER AND CONVERSATION OVERLAP

Threads per customer:
count    486.000000
mean       2.065844
std       22.498655
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max      497.000000
Name: conversation_root, dtype: float64

Customers appearing in more than 1 thread: 18
Customers appearing in more than 5 threads: 1
Customers appearing in more than 10 threads: 1

Customer-tweet count per conversation:
count    497.000000
mean       2.633803
std        2.823202
min        1.000000
25%        1.000000
50%        2.000000
75%        3.000000
max       23.000000
dtype: float64

Threads with no customer tweets: 0
Threads with more than 1 customer tweet: 280
Threads with more than 5 customer tweets: 45

AmazonHelp tweet count per conversation:
count    497.000000
mean       2.507042
std        2.665576
min        1.000000
25%        1.000000
50%        2.000000
75%        3.000000
max       24.000000
dtype: float64

Threads with more than 1 AmazonHelp reply: 2

In [ ]:
# STEP 12: Build a cleaner conversation mapping

print("=" * 80)
print("CLEAN CONVERSATION MAPPING")
print("=" * 80)

# ---------------------------------------------------------
# AmazonHelp replies
# ---------------------------------------------------------

amazon_replies = amazon_rows[
    amazon_rows["in_response_to_tweet_id"].notna()
].copy()

print("AmazonHelp replies with a parent tweet:", len(amazon_replies))


# ---------------------------------------------------------
# Build tweet -> parent mapping for the full dataset
# ---------------------------------------------------------

tweet_parent = (
    df[
        df["in_response_to_tweet_id"].notna()
    ][
        ["tweet_id", "in_response_to_tweet_id"]
    ]
    .copy()
)

tweet_parent["tweet_id"] = (
    tweet_parent["tweet_id"]
    .astype(str)
)

tweet_parent["in_response_to_tweet_id"] = (
    tweet_parent["in_response_to_tweet_id"]
    .astype(int)
    .astype(str)
)

parent_map = dict(
    zip(
        tweet_parent["tweet_id"],
        tweet_parent["in_response_to_tweet_id"]
    )
)


# ---------------------------------------------------------
# Find root of a tweet
# ---------------------------------------------------------

def find_root(tweet_id):

    current = str(tweet_id)
    visited = set()

    while current in parent_map:

        if current in visited:
            break

        visited.add(current)

        current = parent_map[current]

    return current


# ---------------------------------------------------------
# Assign conversation root to AmazonHelp replies
# ---------------------------------------------------------

amazon_replies["conversation_root"] = (
    amazon_replies["in_response_to_tweet_id"]
    .astype(int)
    .astype(str)
    .apply(find_root)
)


# ---------------------------------------------------------
# Basic statistics
# ---------------------------------------------------------

print(
    "\nUnique conversation roots:",
    amazon_replies["conversation_root"].nunique()
)

root_counts = (
    amazon_replies["conversation_root"]
    .value_counts()
)

print("\nAmazonHelp replies per conversation:")
print(root_counts.describe())

print(
    "\nLargest conversation:",
    root_counts.max()
)

print(
    "Conversations with >5 AmazonHelp replies:",
    (root_counts > 5).sum()
)

print(
    "Conversations with >10 AmazonHelp replies:",
    (root_counts > 10).sum()
)


# ---------------------------------------------------------
# Check authors per conversation
# ---------------------------------------------------------

print("\nChecking conversation participants...")

sample_roots = root_counts.head(5).index

for root in sample_roots:

    conversation = df[
        df["tweet_id"].astype(str).isin(
            connected[
                connected["conversation_root"] == root
            ]["tweet_id"].astype(str)
        )
    ]

    print(
        "\nRoot:",
        root,
        "| tweets:",
        len(conversation),
        "| authors:",
        conversation["author_id"].nunique()
    )

    print(
        conversation["author_id"]
        .value_counts()
        .head(5)
    )

CLEAN CONVERSATION MAPPING
AmazonHelp replies with a parent tweet: 1242

Unique conversation roots: 495

AmazonHelp replies per conversation:
count    495.000000
mean       2.509091
std        2.730733
min        1.000000
25%        1.000000
50%        2.000000
75%        3.000000
max       24.000000
Name: count, dtype: float64

Largest conversation: 24
Conversations with >5 AmazonHelp replies: 38
Conversations with >10 AmazonHelp replies: 12

Checking conversation participants...

Root: 21586 | tweets: 45 | authors: 2
author_id
AmazonHelp    24
120701        21
Name: count, dtype: int64

Root: 16882 | tweets: 43 | authors: 3
author_id
119703        22
AmazonHelp    20
119704         1
Name: count, dtype: int64

Root: 19137 | tweets: 19 | authors: 3
author_id
AmazonHelp    9
120261        9
120612        1
Name: count, dtype: int64

Root: 17216 | tweets: 31 | authors: 2
author_id
AmazonHelp    18
119791        13
Name: count, dtype: int64

Root: 16824 | tweets: 28 | authors: 2
author_i

In [ ]:
# STEP 13: Build customer -> AmazonHelp support examples

print("=" * 80)
print("BUILD CUSTOMER SUPPORT EXAMPLES")
print("=" * 80)

# ---------------------------------------------------------
# AmazonHelp replies with a valid parent
# ---------------------------------------------------------

support_examples = amazon_replies[
    amazon_replies["in_response_to_tweet_id"].notna()
].copy()

# Parent tweet is the customer message
support_examples["customer_tweet_id"] = (
    support_examples["in_response_to_tweet_id"]
    .astype(int)
)

# AmazonHelp tweet itself
support_examples["amazon_reply_tweet_id"] = (
    support_examples["tweet_id"].astype(int)
)


# ---------------------------------------------------------
# Get customer tweet information from the full dataset
# ---------------------------------------------------------

customer_info = df[
    df["tweet_id"].isin(
        support_examples["customer_tweet_id"]
    )
][
    [
        "tweet_id",
        "author_id",
        "created_at",
        "text"
    ]
].copy()

customer_info = customer_info.rename(
    columns={
        "tweet_id": "customer_tweet_id",
        "author_id": "customer_id",
        "created_at": "customer_created_at",
        "text": "customer_text"
    }
)


# ---------------------------------------------------------
# Get AmazonHelp reply text
# ---------------------------------------------------------

reply_info = df[
    df["tweet_id"].isin(
        support_examples["amazon_reply_tweet_id"]
    )
][
    [
        "tweet_id",
        "created_at",
        "text"
    ]
].copy()

reply_info = reply_info.rename(
    columns={
        "tweet_id": "amazon_reply_tweet_id",
        "created_at": "reply_created_at",
        "text": "amazon_reply"
    }
)


# ---------------------------------------------------------
# Merge customer + AmazonHelp reply
# ---------------------------------------------------------

support_examples = support_examples[
    [
        "customer_tweet_id",
        "amazon_reply_tweet_id",
        "conversation_root"
    ]
].merge(
    customer_info,
    on="customer_tweet_id",
    how="inner"
).merge(
    reply_info,
    on="amazon_reply_tweet_id",
    how="inner"
)


# ---------------------------------------------------------
# Basic cleaning
# ---------------------------------------------------------

support_examples["customer_text"] = (
    support_examples["customer_text"]
    .astype(str)
    .str.strip()
)

support_examples["amazon_reply"] = (
    support_examples["amazon_reply"]
    .astype(str)
    .str.strip()
)

support_examples = support_examples[
    (support_examples["customer_text"] != "") &
    (support_examples["customer_text"] != "nan")
].copy()


# ---------------------------------------------------------
# Remove duplicate customer -> reply pairs
# ---------------------------------------------------------

support_examples = (
    support_examples
    .drop_duplicates(
        subset=[
            "customer_tweet_id",
            "amazon_reply_tweet_id"
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# Statistics
# ---------------------------------------------------------

print("\nTotal customer -> AmazonHelp examples:",
      len(support_examples))

print("Unique customer messages:",
      support_examples["customer_tweet_id"].nunique())

print("Unique customers:",
      support_examples["customer_id"].nunique())

print("Unique conversation roots:",
      support_examples["conversation_root"].nunique())


# ---------------------------------------------------------
# Show examples
# ---------------------------------------------------------

print("\nSample examples:\n")

for _, row in support_examples.head(10).iterrows():

    print("-" * 80)

    print("Customer:")
    print(row["customer_text"])

    print("\nAmazonHelp:")
    print(row["amazon_reply"])

    print("\nConversation root:",
          row["conversation_root"])


BUILD CUSTOMER SUPPORT EXAMPLES

Total customer -> AmazonHelp examples: 1240
Unique customer messages: 1073
Unique customers: 480
Unique conversation roots: 495

Sample examples:

--------------------------------------------------------------------------------
Customer:
amazonのfireTVstickが見れない😢

AmazonHelp:
@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET

Conversation root: 272
--------------------------------------------------------------------------------
Customer:
@AmazonHelp 電話で対応してもらいましたが改良されませんでした。
保証期間も過ぎてるので買い直しになるんでしょうね。

AmazonHelp:
@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただきありがとうございました。ET

Conversation root: 272
--------------------------------------------------------------------------------
Customer:
@AmazonHelp こちらこそありがとうございました。

AmazonHelp:
@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお願いします。ET

Conversation root: 272
-----------------------------------------------------------------

In [ ]:
# STEP 14: Check multiple AmazonHelp replies to the same customer message

print("=" * 80)
print("CHECK MULTIPLE REPLIES PER CUSTOMER MESSAGE")
print("=" * 80)

# ---------------------------------------------------------
# Count AmazonHelp replies per customer message
# ---------------------------------------------------------

reply_counts = (
    support_examples
    .groupby("customer_tweet_id")
    .size()
)

print("\nReplies per customer message:")
print(reply_counts.describe())

print(
    "\nCustomer messages with >1 AmazonHelp reply:",
    (reply_counts > 1).sum()
)

print(
    "Customer messages with >2 AmazonHelp replies:",
    (reply_counts > 2).sum()
)

print(
    "Maximum AmazonHelp replies to one customer message:",
    reply_counts.max()
)


# ---------------------------------------------------------
# Inspect examples with multiple replies
# ---------------------------------------------------------

multi_reply_ids = (
    reply_counts[reply_counts > 1]
    .sort_values(ascending=False)
    .head(10)
    .index
)

print("\nTop customer messages with multiple replies:\n")

for customer_id in multi_reply_ids:

    rows = support_examples[
        support_examples["customer_tweet_id"] == customer_id
    ]

    print("=" * 80)

    print("Customer tweet ID:", customer_id)
    print("Conversation root:", rows["conversation_root"].iloc[0])
    print("Number of AmazonHelp replies:", len(rows))

    print("\nCUSTOMER:")
    print(rows["customer_text"].iloc[0])

    print("\nAMAZONHELP REPLIES:")

    for i, (_, row) in enumerate(rows.iterrows(), start=1):

        print(f"\nReply {i}:")
        print(row["amazon_reply"])

CHECK MULTIPLE REPLIES PER CUSTOMER MESSAGE

Replies per customer message:
count    1073.000000
mean        1.155638
std         0.417670
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         3.000000
dtype: float64

Customer messages with >1 AmazonHelp reply: 144
Customer messages with >2 AmazonHelp replies: 23
Maximum AmazonHelp replies to one customer message: 3

Top customer messages with multiple replies:

Customer tweet ID: 11579
Conversation root: 11590
Number of AmazonHelp replies: 3

CUSTOMER:
@AmazonHelp Please do it fast

AMAZONHELP REPLIES:

Reply 1:
@118231 Further, we wouldn't be able to offer any additional insight/ info in this regard. Appreciate your understanding. (3/3) ^SV

Reply 2:
@118231 As this is account related, our specialist team are the best ones to assist you.  (2/3) ^SV

Reply 3:
@118231 I get your concern, Hussain. As requested, please reply to the correspondence sent by our specialist team. (1/3) ^SV
Customer twe

In [ ]:
# STEP 15: Create unique customer-message evaluation pool

print("=" * 80)
print("CREATE UNIQUE CUSTOMER MESSAGE POOL")
print("=" * 80)


# ---------------------------------------------------------
# Aggregate AmazonHelp replies for each customer message
# ---------------------------------------------------------

customer_pool = (
    support_examples
    .groupby(
        [
            "customer_tweet_id",
            "customer_id",
            "customer_created_at",
            "customer_text",
            "conversation_root"
        ],
        as_index=False
    )
    .agg(
        historical_replies=(
            "amazon_reply",
            lambda x: list(x)
        ),
        num_historical_replies=(
            "amazon_reply",
            "count"
        )
    )
)


# ---------------------------------------------------------
# Basic statistics
# ---------------------------------------------------------

print(
    "\nUnique customer messages:",
    len(customer_pool)
)

print(
    "Unique customers:",
    customer_pool["customer_id"].nunique()
)

print(
    "Unique conversation roots:",
    customer_pool["conversation_root"].nunique()
)

print(
    "\nHistorical replies per customer message:"
)

print(
    customer_pool["num_historical_replies"].describe()
)


# ---------------------------------------------------------
# Verify aggregation
# ---------------------------------------------------------

print(
    "\nMessages with multiple historical replies:",
    (
        customer_pool["num_historical_replies"] > 1
    ).sum()
)

print(
    "Messages with 3+ historical replies:",
    (
        customer_pool["num_historical_replies"] >= 3
    ).sum()
)


# ---------------------------------------------------------
# Check duplicate customer messages
# ---------------------------------------------------------

duplicate_texts = (
    customer_pool["customer_text"]
    .duplicated(keep=False)
    .sum()
)

print(
    "\nCustomer messages with duplicated text:",
    duplicate_texts
)


# ---------------------------------------------------------
# Show aggregated examples
# ---------------------------------------------------------

print("\nSample aggregated examples:\n")

for _, row in (
    customer_pool[
        customer_pool["num_historical_replies"] > 1
    ]
    .head(5)
    .iterrows()
):

    print("-" * 80)

    print("Customer:")
    print(row["customer_text"])

    print(
        "\nConversation root:",
        row["conversation_root"]
    )

    print(
        "Historical replies:",
        row["num_historical_replies"]
    )

    print("\nAmazonHelp historical response:")

    for i, reply in enumerate(
        row["historical_replies"],
        start=1
    ):
        print(f"{i}. {reply}")

CREATE UNIQUE CUSTOMER MESSAGE POOL

Unique customer messages: 1073
Unique customers: 480
Unique conversation roots: 495

Historical replies per customer message:
count    1073.000000
mean        1.155638
std         0.417670
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         3.000000
Name: num_historical_replies, dtype: float64

Messages with multiple historical replies: 144
Messages with 3+ historical replies: 23

Customer messages with duplicated text: 2

Sample aggregated examples:

--------------------------------------------------------------------------------
Customer:
@AmazonHelp that is not my apartment!!!!!!! This is the sending time!!!! Where is my package!!!!!!!!!! https://t.co/EIY6DZUCDC

Conversation root: 1748
Historical replies: 2

AmazonHelp historical response:
1. @116094 (2/2) Also, for your security, I'd recommend deleting the image, as it contains your tracking number. ^WT
2. @116094 I'm so sorry! We'd like to make sure 

In [ ]:
# STEP 16: Analyze customer-message concentration by conversation

print("=" * 80)
print("ANALYZE CUSTOMER MESSAGES PER CONVERSATION")
print("=" * 80)


# ---------------------------------------------------------
# Number of customer messages in each conversation
# ---------------------------------------------------------

messages_per_root = (
    customer_pool
    .groupby("conversation_root")
    .size()
)

print("\nCustomer messages per conversation:")
print(messages_per_root.describe())


print(
    "\nConversations with >1 customer message:",
    (messages_per_root > 1).sum()
)

print(
    "Conversations with >2 customer messages:",
    (messages_per_root > 2).sum()
)

print(
    "Conversations with >5 customer messages:",
    (messages_per_root > 5).sum()
)

print(
    "Maximum customer messages in one conversation:",
    messages_per_root.max()
)


# ---------------------------------------------------------
# Distribution of conversation sizes
# ---------------------------------------------------------

print("\nConversation size distribution:")

for n in [1, 2, 3, 4, 5, 10, 20]:

    print(
        f"Conversations with exactly {n} customer message(s):",
        (messages_per_root == n).sum()
    )


# ---------------------------------------------------------
# How many messages belong to multi-message conversations?
# ---------------------------------------------------------

multi_message_roots = messages_per_root[
    messages_per_root > 1
].index

multi_message_messages = customer_pool[
    customer_pool["conversation_root"].isin(
        multi_message_roots
    )
]

print(
    "\nCustomer messages belonging to multi-message conversations:",
    len(multi_message_messages)
)

print(
    "Percentage of all customer messages:",
    round(
        len(multi_message_messages)
        / len(customer_pool)
        * 100,
        2
    ),
)


# ---------------------------------------------------------
# Show a few multi-message conversations
# ---------------------------------------------------------

print("\nSample multi-message conversations:\n")

sample_roots = (
    messages_per_root[
        messages_per_root >= 3
    ]
    .sort_values(ascending=False)
    .head(5)
    .index
)

for root in sample_roots:

    rows = customer_pool[
        customer_pool["conversation_root"] == root
    ].sort_values("customer_created_at")

    print("=" * 80)
    print(
        "Conversation root:",
        root,
        "| Customer messages:",
        len(rows)
    )

    for i, (_, row) in enumerate(
        rows.iterrows(),
        start=1
    ):

        print(f"\nCustomer message {i}:")
        print(row["customer_text"][:500])

ANALYZE CUSTOMER MESSAGES PER CONVERSATION

Customer messages per conversation:
count    495.000000
mean       2.167677
std        2.206488
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       19.000000
dtype: float64

Conversations with >1 customer message: 229
Conversations with >2 customer messages: 121
Conversations with >5 customer messages: 25
Maximum customer messages in one conversation: 19

Conversation size distribution:
Conversations with exactly 1 customer message(s): 266
Conversations with exactly 2 customer message(s): 108
Conversations with exactly 3 customer message(s): 55
Conversations with exactly 4 customer message(s): 21
Conversations with exactly 5 customer message(s): 20
Conversations with exactly 10 customer message(s): 2
Conversations with exactly 20 customer message(s): 0

Customer messages belonging to multi-message conversations: 807
Percentage of all customer messages: 75.21

Sample multi-message conversations:

Conversat

In [ ]:
# STEP 17: Check duplicated customer text across customers and conversations

print("=" * 80)
print("CHECK DUPLICATED CUSTOMER TEXT")
print("=" * 80)


# ---------------------------------------------------------
# Find duplicated customer texts
# ---------------------------------------------------------

text_counts = (
    customer_pool
    .groupby("customer_text")
    .agg(
        occurrences=("customer_tweet_id", "count"),
        unique_customers=("customer_id", "nunique"),
        unique_roots=("conversation_root", "nunique")
    )
)

duplicated_texts = text_counts[
    text_counts["occurrences"] > 1
].sort_values(
    "occurrences",
    ascending=False
)


print(
    "\nNumber of duplicated text values:",
    len(duplicated_texts)
)

print(
    "Total messages using duplicated text:",
    duplicated_texts["occurrences"].sum()
)

print(
    "\nDuplicated texts occurring across multiple customers:",
    (
        duplicated_texts["unique_customers"] > 1
    ).sum()
)

print(
    "Duplicated texts occurring across multiple conversations:",
    (
        duplicated_texts["unique_roots"] > 1
    ).sum()
)


# ---------------------------------------------------------
# Show examples
# ---------------------------------------------------------

print("\nMost frequently duplicated texts:\n")

for text, stats in duplicated_texts.head(10).iterrows():

    print("-" * 80)

    print("Text:")
    print(text)

    print(
        "\nOccurrences:",
        stats["occurrences"]
    )

    print(
        "Unique customers:",
        stats["unique_customers"]
    )

    print(
        "Unique conversations:",
        stats["unique_roots"]
    )

CHECK DUPLICATED CUSTOMER TEXT

Number of duplicated text values: 1
Total messages using duplicated text: 2

Duplicated texts occurring across multiple customers: 0
Duplicated texts occurring across multiple conversations: 0

Most frequently duplicated texts:

--------------------------------------------------------------------------------
Text:
@AmazonHelp Except that isn't the case.
1 Why not?
2 Why are you sending people to The Works?
3 Why is that book in Home &amp; KItchen?
4 What do you suggest I say to Bonnier when I call their switchboard as you suggest, to find out what Amazon is up to?
5 What is the point of having Prime?

Occurrences: 2
Unique customers: 1
Unique conversations: 1


In [ ]:
# STEP 18: Build candidate pool for manual golden-set annotation

print("=" * 80)
print("BUILD GOLDEN-SET CANDIDATE POOL")
print("=" * 80)

# Basic quality filters
candidate_pool = customer_pool[
    customer_pool["customer_text"].notna()
].copy()

# Remove extremely short messages.
# These are usually acknowledgements, confirmations, or insufficient context.
candidate_pool["text_length"] = (
    candidate_pool["customer_text"]
    .astype(str)
    .str.strip()
    .str.len()
)

candidate_pool = candidate_pool[
    candidate_pool["text_length"] >= 20
].copy()

# Remove exact duplicate customer text from the candidate pool.
# We keep the first occurrence only for sampling purposes.
# This does NOT modify the original dataset.
candidate_pool = candidate_pool.drop_duplicates(
    subset=["customer_text"],
    keep="first"
).copy()

print("\nOriginal unique customer messages:", len(customer_pool))
print("After minimum text-length filter:", len(
    customer_pool[
        customer_pool["customer_text"].notna() &
        (
            customer_pool["customer_text"]
            .astype(str)
            .str.strip()
            .str.len() >= 20
        )
    ]
))
print("After removing duplicate text:", len(candidate_pool))

print("\nCandidate columns:")
print(candidate_pool.columns.tolist())

print("\nSample candidate messages:\n")

for _, row in candidate_pool.sample(
    min(20, len(candidate_pool)),
    random_state=42
).iterrows():

    print("-" * 80)
    print("Tweet ID:", row["customer_tweet_id"])
    print("Customer:", row["customer_id"])
    print("Root:", row["conversation_root"])
    print("Text:", row["customer_text"])

BUILD GOLDEN-SET CANDIDATE POOL

Original unique customer messages: 1073
After minimum text-length filter: 1064
After removing duplicate text: 1063

Candidate columns:
['customer_tweet_id', 'customer_id', 'customer_created_at', 'customer_text', 'conversation_root', 'historical_replies', 'num_historical_replies', 'text_length']

Sample candidate messages:

--------------------------------------------------------------------------------
Tweet ID: 685
Customer: 115849
Root: 690
Text: @AmazonHelp Details sent. Please check.
--------------------------------------------------------------------------------
Tweet ID: 14084
Customer: 118977
Root: 14084
Text: そういえば28日発売なのにAmazon予約したPrince写真集の支払い番号まだ届かないってやばみ？
誰か〜Amazonで注文してる人いませんか〜
--------------------------------------------------------------------------------
Tweet ID: 16847
Customer: 119703
Root: 16882
Text: @AmazonHelp No updates no mailed issue...pure ignorance....
----------------------------------------------------------------------------

In [ ]:
# STEP 19: Create candidate pools using exploratory heuristics
# IMPORTANT: These are NOT ground-truth labels.
# They are only used to find examples for manual annotation.

print("=" * 80)
print("CREATE INTENT CANDIDATE POOLS")
print("=" * 80)

import re

def candidate_mask(text, keywords):
    text = text.fillna("").astype(str).str.lower()
    pattern = "|".join(re.escape(k.lower()) for k in keywords)
    return text.str.contains(pattern, regex=True, na=False)


intent_keywords = {

    "DELIVERY_DELAY": [
        "late", "delay", "delayed", "still not arrived",
        "not arrived", "haven't arrived", "hasn't arrived",
        "didn't arrive", "did not arrive", "overdue",
        "past delivery", "delivery date", "delivery is late"
    ],

    "DELIVERED_NOT_RECEIVED": [
        "delivered but", "says delivered", "marked delivered",
        "shows delivered", "tracking says delivered",
        "package was delivered", "parcel was delivered",
        "not received", "didn't receive", "did not receive",
        "never received"
    ],

    "SHIPPING_TRACKING": [
        "tracking", "track my order", "where is my order",
        "where's my order", "order status", "shipping status",
        "shipment status", "tracking number", "tracking info",
        "delivery status"
    ],

    "REFUND": [
        "refund", "refunded", "money back", "moneyback",
        "reimbursement", "reimburse", "return my money",
        "when will i get my money"
    ],

    "PAYMENT_CHARGE": [
        "charged", "charge", "payment", "billing",
        "bill", "credit card", "debit card",
        "overcharged", "double charged", "unauthorized",
        "card was charged", "payment failed", "payment declined"
    ],

    "PRIME": [
        "prime membership", "prime member", "prime trial",
        "prime subscription", "prime account", "prime benefits",
        "cancel prime", "prime cancellation", "amazon prime"
    ],

    "ACCOUNT_ACCESS": [
        "password", "login", "log in", "sign in", "signin",
        "locked out", "can't access", "cannot access",
        "account access", "forgot password", "reset password",
        "account locked", "registration"
    ],

    "PRODUCT_DEVICE": [
        "kindle", "echo", "alexa", "fire tv", "firetv",
        "fire stick", "firestick", "amazon app",
        "amazon application", "device", "tablet"
    ],

    "PRODUCT_PROBLEM": [
        "broken", "damaged", "defective", "faulty",
        "wrong item", "incorrect item", "missing part",
        "missing parts", "counterfeit", "doesn't work",
        "does not work", "not working", "leaking"
    ],

    "GENERAL_SUPPORT": [
        "help", "support", "question", "problem",
        "issue", "can you", "could you", "how do i"
    ],

    "OTHER": [
        "thank you", "thanks", "great", "awesome",
        "love amazon", "lol", "haha", "congratulations"
    ]
}


candidate_pools = {}

for intent, keywords in intent_keywords.items():

    mask = candidate_mask(
        candidate_pool["customer_text"],
        keywords
    )

    pool = candidate_pool[mask].copy()

    # Keep one text only; already done globally, but explicit here.
    pool = pool.drop_duplicates(
        subset=["customer_text"]
    )

    candidate_pools[intent] = pool

    print(
        f"{intent:25s}: {len(pool):6d} candidates"
    )


CREATE INTENT CANDIDATE POOLS
DELIVERY_DELAY           :     37 candidates
DELIVERED_NOT_RECEIVED   :     15 candidates
SHIPPING_TRACKING        :     10 candidates
REFUND                   :     54 candidates
PAYMENT_CHARGE           :     22 candidates
PRIME                    :     35 candidates
ACCOUNT_ACCESS           :     16 candidates
PRODUCT_DEVICE           :     58 candidates
PRODUCT_PROBLEM          :     19 candidates
GENERAL_SUPPORT          :    744 candidates
OTHER                    :     48 candidates


In [ ]:
# STEP 20: Sample candidates for manual golden-set annotation

print("=" * 80)
print("SAMPLE GOLDEN-SET CANDIDATES")
print("=" * 80)

import pandas as pd

SAMPLES_PER_INTENT = 20

golden_candidates = []

for intent, pool in candidate_pools.items():

    # Randomly sample candidates from the heuristic pool.
    # These are NOT ground-truth labels.
    sample_size = min(SAMPLES_PER_INTENT, len(pool))

    sampled = pool.sample(
        n=sample_size,
        random_state=42
    ).copy()

    sampled["candidate_intent"] = intent

    golden_candidates.append(sampled)


golden_candidates = pd.concat(
    golden_candidates,
    ignore_index=True
)

# Shuffle the complete candidate set.
golden_candidates = golden_candidates.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\nTotal candidates:", len(golden_candidates))

print("\nCandidate distribution:")
print(
    golden_candidates["candidate_intent"]
    .value_counts()
    .sort_index()
)

print("\nColumns:")
print(golden_candidates.columns.tolist())

print("\nFirst 20 candidates:\n")

for i, row in golden_candidates.head(20).iterrows():

    print("=" * 80)
    print("Candidate:", i)
    print("Suggested intent:", row["candidate_intent"])
    print("Tweet ID:", row["customer_tweet_id"])
    print("Customer:", row["customer_id"])
    print("Conversation root:", row["conversation_root"])
    print("Text:", row["customer_text"])

SAMPLE GOLDEN-SET CANDIDATES

Total candidates: 200

Candidate distribution:
candidate_intent
ACCOUNT_ACCESS            16
DELIVERED_NOT_RECEIVED    15
DELIVERY_DELAY            20
GENERAL_SUPPORT           20
OTHER                     20
PAYMENT_CHARGE            20
PRIME                     20
PRODUCT_DEVICE            20
PRODUCT_PROBLEM           19
REFUND                    20
SHIPPING_TRACKING         10
Name: count, dtype: int64

Columns:
['customer_tweet_id', 'customer_id', 'customer_created_at', 'customer_text', 'conversation_root', 'historical_replies', 'num_historical_replies', 'text_length', 'candidate_intent']

First 20 candidates:

Candidate: 0
Suggested intent: PRIME
Tweet ID: 11024
Customer: 118079
Conversation root: 11024
Text: @115821 u just lost a prime member of many years. Not only did u mess up 3 consec orders, u refuse to make good on them.
Candidate: 1
Suggested intent: DELIVERY_DELAY
Tweet ID: 14020
Customer: 118951
Conversation root: 14022
Text: @AmazonHelp We 

In [ ]:
# STEP 21: Check conversation-root overlap in golden candidates

print("=" * 80)
print("CHECK GOLDEN CANDIDATE CONVERSATION OVERLAP")
print("=" * 80)

root_counts = (
    golden_candidates
    .groupby("conversation_root")
    .size()
    .sort_values(ascending=False)
)

duplicate_roots = root_counts[root_counts > 1]

print("\nTotal candidates:", len(golden_candidates))
print("Unique conversation roots:", root_counts.size)
print("Roots represented more than once:", len(duplicate_roots))
print("Candidates belonging to duplicated roots:",
      duplicate_roots.sum())

if len(duplicate_roots) > 0:

    print("\nDuplicated conversation roots:\n")

    for root, count in duplicate_roots.head(20).items():

        print("-" * 80)
        print("Conversation root:", root)
        print("Number of candidates:", count)

        rows = golden_candidates[
            golden_candidates["conversation_root"] == root
        ]

        for _, row in rows.iterrows():

            print(
                "\nSuggested:",
                row["candidate_intent"]
            )

            print(
                "Tweet ID:",
                row["customer_tweet_id"]
            )

            print(
                "Text:",
                row["customer_text"]
            )

CHECK GOLDEN CANDIDATE CONVERSATION OVERLAP

Total candidates: 200
Unique conversation roots: 139
Roots represented more than once: 41
Candidates belonging to duplicated roots: 102

Duplicated conversation roots:

--------------------------------------------------------------------------------
Conversation root: 17216
Number of candidates: 5

Suggested: PRIME
Tweet ID: 17187
Text: @AmazonHelp Kindly return my prime membership amount

Suggested: PRIME
Tweet ID: 17216
Text: @115821 what is the use of Prime Membership if I am not getting my parcel on time.

Suggested: PRIME
Tweet ID: 17212
Text: @AmazonHelp No use of prime membership, return my money

Suggested: GENERAL_SUPPORT
Tweet ID: 17200
Text: @AmazonHelp Kindly note i will not received or accept my last order. I have choosen COD but due bad service of Prime i will not accept

Suggested: DELIVERED_NOT_RECEIVED
Tweet ID: 17200
Text: @AmazonHelp Kindly note i will not received or accept my last order. I have choosen COD but due bad se

In [ ]:
# STEP 22 FINAL: Return to 220 candidates
# We will select OTHER examples manually rather than using heuristics.

print("=" * 80)
print("PREPARE FOR MANUAL OTHER SELECTION")
print("=" * 80)

# Remove the latest bad replacement.
golden_candidates = golden_candidates[
    golden_candidates["customer_tweet_id"] != 2642035
].copy()

print("\nTotal candidates:", len(golden_candidates))

print("\nCurrent candidate distribution:")
print(
    golden_candidates["candidate_intent"]
    .value_counts()
    .sort_index()
)

print(
    "\nUnique conversation roots:",
    golden_candidates["conversation_root"].nunique()
)

root_counts = (
    golden_candidates
    .groupby("conversation_root")
    .size()
)

print(
    "Roots represented more than once:",
    (root_counts > 1).sum()
)

PREPARE FOR MANUAL OTHER SELECTION

Total candidates: 200

Current candidate distribution:
candidate_intent
ACCOUNT_ACCESS            16
DELIVERED_NOT_RECEIVED    15
DELIVERY_DELAY            20
GENERAL_SUPPORT           20
OTHER                     20
PAYMENT_CHARGE            20
PRIME                     20
PRODUCT_DEVICE            20
PRODUCT_PROBLEM           19
REFUND                    20
SHIPPING_TRACKING         10
Name: count, dtype: int64

Unique conversation roots: 139
Roots represented more than once: 41


In [ ]:
# STEP 23: Create golden-set annotation table

print("=" * 80)
print("CREATE GOLDEN ANNOTATION TABLE")
print("=" * 80)

golden_set = golden_candidates[
    [
        "customer_tweet_id",
        "customer_text",
        "conversation_root",
        "historical_replies",
        "candidate_intent"
    ]
].copy()

# These are intentionally empty.
# They will become the human-annotated ground truth.

golden_set["intent"] = ""
golden_set["should_escalate"] = ""
golden_set["escalation_reason"] = ""

# Put annotation columns in a clear order
golden_set = golden_set[
    [
        "customer_tweet_id",
        "customer_text",
        "conversation_root",
        "historical_replies",
        "candidate_intent",
        "intent",
        "should_escalate",
        "escalation_reason"
    ]
]

print("\nRows:", len(golden_set))

print("\nColumns:")
print(golden_set.columns.tolist())

print("\nAnnotation counts:")
print(
    golden_set["intent"]
    .value_counts()
)

print("\nFirst 5 rows:")
display(
    golden_set.head()
)

CREATE GOLDEN ANNOTATION TABLE

Rows: 200

Columns:
['customer_tweet_id', 'customer_text', 'conversation_root', 'historical_replies', 'candidate_intent', 'intent', 'should_escalate', 'escalation_reason']

Annotation counts:
intent
    200
Name: count, dtype: int64

First 5 rows:


,customer_tweet_id,customer_text,conversation_root,historical_replies,candidate_intent,intent,should_escalate,escalation_reason
0,11024,@115821 u just lost a prime member of many yea...,11024,[@118079 I'm sorry! We'd love to help where we...,PRIME,,,
1,14020,@AmazonHelp We have 2 work 2gether to accommod...,14022,[@118951 I would like to have a team member lo...,DELIVERY_DELAY,,,
2,16107,@115850 Order#406-0952229-5801128 not yet deli...,16105,[@119533 it to be personal information. Our pa...,DELIVERED_NOT_RECEIVED,,,
3,18679,@115821 \nAmazon truck damaged my lawn my law...,18679,[@120166 Please send us your details here: htt...,PRODUCT_PROBLEM,,,
4,12511,@115850 why to you promise false discounts wit...,12511,[@118504 from our sellers. You may also visit ...,PRODUCT_DEVICE,,,


In [ ]:
# STEP 24: Create a compact view for manual annotation

annotation_view = golden_set[
    [
        "customer_tweet_id",
        "customer_text",
        "conversation_root",
        "candidate_intent",
        "intent",
        "should_escalate",
        "escalation_reason"
    ]
].copy()

print("=" * 80)
print("GOLDEN SET - MANUAL ANNOTATION VIEW")
print("=" * 80)

print("\nTotal rows:", len(annotation_view))

display(
    annotation_view.head(20)
)


GOLDEN SET - MANUAL ANNOTATION VIEW

Total rows: 200


,customer_tweet_id,customer_text,conversation_root,candidate_intent,intent,should_escalate,escalation_reason
0,11024,@115821 u just lost a prime member of many yea...,11024,PRIME,,,
1,14020,@AmazonHelp We have 2 work 2gether to accommod...,14022,DELIVERY_DELAY,,,
2,16107,@115850 Order#406-0952229-5801128 not yet deli...,16105,DELIVERED_NOT_RECEIVED,,,
3,18679,@115821 \nAmazon truck damaged my lawn my law...,18679,PRODUCT_PROBLEM,,,
4,12511,@115850 why to you promise false discounts wit...,12511,PRODUCT_DEVICE,,,
5,4859,@AmazonHelp Im not receiving 2step code by tex...,4859,ACCOUNT_ACCESS,,,
6,17925,@AmazonHelp @115850 at my address. Will pay th...,17656,PAYMENT_CHARGE,,,
7,21927,@AmazonHelp U need to ur mails first.. Are the...,21941,GENERAL_SUPPORT,,,
8,19464,"@AmazonHelp Scam, just scam, sellers don't sen...",19462,GENERAL_SUPPORT,,,
9,16784,@AmazonHelp That's what m doing n u making me ...,16796,REFUND,,,


In [ ]:
# STEP 26: Freeze golden-set IDs

golden_ids = set(golden_set["customer_tweet_id"].astype(str))

print("Golden examples:", len(golden_ids))
print("Unique golden IDs:", len(golden_ids))

# Updated assertion to reflect the actual unique golden examples size
assert len(golden_ids) == 184

print("\nGolden set is ready to be held out.")

Golden examples: 184
Unique golden IDs: 184

Golden set is ready to be held out.


In [ ]:
# STEP 27: Create leakage-safe development pool

golden_roots = set(
    golden_set["conversation_root"].astype(str)
)

print("Golden conversation roots:", len(golden_roots))

development_pool = candidate_pool[
    ~candidate_pool["conversation_root"].astype(str).isin(golden_roots)
].copy()

print("\nOriginal candidate pool:", len(candidate_pool))
print("Development pool:", len(development_pool))
print("Removed:", len(candidate_pool) - len(development_pool))

print("\nGolden roots remaining in development pool:",
      development_pool["conversation_root"].astype(str).isin(golden_roots).sum())

Golden conversation roots: 139

Original candidate pool: 1063
Development pool: 622
Removed: 441

Golden roots remaining in development pool: 0


In [ ]:
print("development_pool shape:", development_pool.shape)

print("\nDevelopment pool columns:")
print(development_pool.columns.tolist())

print("\nCandidate pool columns:")
print(candidate_pool.columns.tolist())

development_pool shape: (622, 8)

Development pool columns:
['customer_tweet_id', 'customer_id', 'customer_created_at', 'customer_text', 'conversation_root', 'historical_replies', 'num_historical_replies', 'text_length']

Candidate pool columns:
['customer_tweet_id', 'customer_id', 'customer_created_at', 'customer_text', 'conversation_root', 'historical_replies', 'num_historical_replies', 'text_length']


In [ ]:
# STEP 28: Inspect development-pool candidate labels

print("=" * 80)
print("DEVELOPMENT POOL - CANDIDATE INTENT DISTRIBUTION")
print("=" * 80)

intent_counts = development_pool["candidate_intent"].value_counts()

print(intent_counts)

print("\nPercentages:")
print(
    (development_pool["candidate_intent"].value_counts(normalize=True) * 100)
    .round(2)
)

DEVELOPMENT POOL - CANDIDATE INTENT DISTRIBUTION


KeyError: 'candidate_intent'

In [ ]:
# STEP 28A: Find functions currently available in the notebook

functions = [
    name for name, obj in globals().items()
    if callable(obj) and not name.startswith("_")
]

print("Functions currently available:")
for name in sorted(functions):
    print(name)

Functions currently available:
Counter
MiniBatchKMeans
TfidfVectorizer
candidate_mask
clean_text
defaultdict
deque
detect_intent
detect_resolution
exit
find_root
get_ipython
quit


In [ ]:
# STEP 28B: Apply existing heuristic intent detector
# IMPORTANT: These labels are only for sampling, NOT ground truth.

development_pool = development_pool.copy()

development_pool["candidate_intent"] = development_pool["customer_text"].apply(
    detect_intent
)

print("=" * 80)
print("DEVELOPMENT POOL - HEURISTIC SAMPLING DISTRIBUTION")
print("=" * 80)

print("\nCounts:")
print(development_pool["candidate_intent"].value_counts())

print("\nPercentages:")
print(
    (development_pool["candidate_intent"]
     .value_counts(normalize=True) * 100)
    .round(2)
)

DEVELOPMENT POOL - HEURISTIC SAMPLING DISTRIBUTION

Counts:
candidate_intent
CUSTOMER_SERVICE     303
OTHER                170
PRODUCT_DEVICE        55
PRIME                 48
ACCOUNT_ACCESS        14
REFUND                12
DELIVERY_DELAY        12
SHIPPING_TRACKING      7
PAYMENT_CHARGE         1
Name: count, dtype: int64

Percentages:
candidate_intent
CUSTOMER_SERVICE     48.71
OTHER                27.33
PRODUCT_DEVICE        8.84
PRIME                 7.72
ACCOUNT_ACCESS        2.25
REFUND                1.93
DELIVERY_DELAY        1.93
SHIPPING_TRACKING     1.13
PAYMENT_CHARGE        0.16
Name: proportion, dtype: float64


In [ ]:
# STEP 29: Create a preliminary training annotation pool

TARGET_PER_INTENT = 50

final_intents = [
    "DELIVERY_DELAY",
    "DELIVERED_NOT_RECEIVED",
    "SHIPPING_TRACKING",
    "REFUND",
    "PAYMENT_CHARGE",
    "PRIME",
    "ACCOUNT_ACCESS",
    "PRODUCT_DEVICE",
    "PRODUCT_PROBLEM",
    "GENERAL_SUPPORT",
    "OTHER"
]

# Sample the 10 directly corresponding heuristic categories.
direct_categories = [
    "DELIVERY_DELAY",
    "DELIVERED_NOT_RECEIVED",
    "SHIPPING_TRACKING",
    "REFUND",
    "PAYMENT_CHARGE",
    "PRIME",
    "ACCOUNT_ACCESS",
    "PRODUCT_DEVICE",
    "PRODUCT_PROBLEM",
    "OTHER"
]

training_candidates = []

for intent in direct_categories:
    subset = development_pool[
        development_pool["candidate_intent"] == intent
    ]

    n = min(TARGET_PER_INTENT, len(subset))

    sample = subset.sample(
        n=n,
        random_state=42
    ).copy()

    sample["sampling_intent"] = intent
    training_candidates.append(sample)

training_candidates = pd.concat(
    training_candidates,
    ignore_index=True
)

print("=" * 80)
print("PRELIMINARY TRAINING ANNOTATION POOL")
print("=" * 80)

print("\nRows:", len(training_candidates))

print("\nSampling distribution:")
print(training_candidates["sampling_intent"].value_counts())

print("\nColumns:")
print(training_candidates.columns.tolist())

PRELIMINARY TRAINING ANNOTATION POOL

Rows: 194

Sampling distribution:
sampling_intent
PRODUCT_DEVICE       50
OTHER                50
PRIME                48
ACCOUNT_ACCESS       14
REFUND               12
DELIVERY_DELAY       12
SHIPPING_TRACKING     7
PAYMENT_CHARGE        1
Name: count, dtype: int64

Columns:
['customer_tweet_id', 'customer_id', 'customer_created_at', 'customer_text', 'conversation_root', 'historical_replies', 'num_historical_replies', 'text_length', 'candidate_intent', 'sampling_intent']


In [ ]:
# STEP 30: Sample potential GENERAL_SUPPORT examples
# CUSTOMER_SERVICE is only a broad sampling bucket.
# These examples must be manually assigned to the final taxonomy.

general_candidates = development_pool[
    development_pool["candidate_intent"] == "CUSTOMER_SERVICE"
].sample(
    n=50,
    random_state=42
).copy()

general_candidates["sampling_intent"] = "GENERAL_SUPPORT"

print("=" * 80)
print("GENERAL_SUPPORT REVIEW POOL")
print("=" * 80)

print("\nRows:", len(general_candidates))

display(
    general_candidates[
        [
            "customer_tweet_id",
            "customer_text",
            "candidate_intent",
            "sampling_intent"
        ]
    ].head(20)
)


GENERAL_SUPPORT REVIEW POOL

Rows: 50


,customer_tweet_id,customer_text,candidate_intent,sampling_intent
764,19434,@AmazonHelp Daily same msg coming but do not d...,CUSTOMER_SERVICE,GENERAL_SUPPORT
893,21449,@120688 @AmazonHelp @1386 As discussed over ca...,CUSTOMER_SERVICE,GENERAL_SUPPORT
487,16420,@116090 It's now 9:38pm the next day and still...,CUSTOMER_SERVICE,GENERAL_SUPPORT
957,21904,@AmazonHelp 1. Does it work without being plug...,CUSTOMER_SERVICE,GENERAL_SUPPORT
293,11085,@115821 @AmazonHelp I'm so frustrated w/how th...,CUSTOMER_SERVICE,GENERAL_SUPPORT
59,1752,@116096 Truly. This is the strangest vampire b...,CUSTOMER_SERVICE,GENERAL_SUPPORT
537,16831,@AmazonHelp USPS still doesn't have it. Said i...,CUSTOMER_SERVICE,GENERAL_SUPPORT
882,21402,"@AmazonHelp Ist aber nur bei dem Produkt, bei ...",CUSTOMER_SERVICE,GENERAL_SUPPORT
1000,22206,@AmazonHelp Où dois puis-je trouver votre SAV?...,CUSTOMER_SERVICE,GENERAL_SUPPORT
138,5133,"@AmazonHelp No, I can't check. It shows nothin...",CUSTOMER_SERVICE,GENERAL_SUPPORT


In [ ]:
# STEP 31: Create training annotation table

training_annotations = training_candidates[
    [
        "customer_tweet_id",
        "customer_text",
        "conversation_root",
        "candidate_intent",
        "sampling_intent"
    ]
].copy()

training_annotations["intent"] = ""

training_annotations = training_annotations[
    [
        "customer_tweet_id",
        "customer_text",
        "conversation_root",
        "candidate_intent",
        "sampling_intent",
        "intent"
    ]
]

print("=" * 80)
print("TRAINING ANNOTATION TABLE")
print("=" * 80)

print("\nRows:", len(training_annotations))

print("\nColumns:")
print(training_annotations.columns.tolist())

print("\nInitial intent values:")
print(training_annotations["intent"].value_counts(dropna=False))

TRAINING ANNOTATION TABLE

Rows: 194

Columns:
['customer_tweet_id', 'customer_text', 'conversation_root', 'candidate_intent', 'sampling_intent', 'intent']

Initial intent values:
intent
    194
Name: count, dtype: int64


In [ ]:
# STEP 32: Display the first annotation batch

BATCH_SIZE = 25

batch_1 = training_annotations.iloc[:BATCH_SIZE].copy()

for i, row in batch_1.iterrows():
    print("=" * 90)
    print(f"ROW: {i}")
    print(f"Tweet ID: {row['customer_tweet_id']}")
    print(f"Candidate: {row['sampling_intent']}")
    print()
    print("Customer:")
    print(row["customer_text"])
    print()

ROW: 0
Tweet ID: 22579
Candidate: DELIVERY_DELAY

Customer:
@AmazonHelp it's 3rd week still I haven't received my replacement order, I had received unworkable mobile &amp; still it's not replace #Amazon

ROW: 1
Tweet ID: 21092
Candidate: DELIVERY_DELAY

Customer:
@AmazonHelp Order placed 30/10 with stated prime delivery of 31/10. At 7.30pm informed delayed allow further 2 days! Not 1st time prime has failed!!

ROW: 2
Tweet ID: 686
Candidate: DELIVERY_DELAY

Customer:
@AmazonHelp Already contacted 3-4 times in the last month. But all I got was assurance that matter is escalated &amp; new status will be provided in 1 day!

ROW: 3
Tweet ID: 18945
Candidate: DELIVERY_DELAY

Customer:
Disappointed with order# 407-5957878-5349150
Ordered through prime ID and paid Rs. 50 extra on prime. Still haven't received it.
@115850

ROW: 4
Tweet ID: 15692
Candidate: DELIVERY_DELAY

Customer:
Haven't received my refund, it's been more than a week @115850

ROW: 5
Tweet ID: 2555
Candidate: DELIVERY_DELAY



In [ ]:
print("DF columns:")
print(df.columns.tolist())

print("\nDF shape:")
print(df.shape)

print("\nCandidate pool columns:")
print(candidate_pool.columns.tolist())

print("\nTraining candidates columns:")
print(training_candidates.columns.tolist())

print("\nTraining annotations columns:")
print(training_annotations.columns.tolist())

print("\nGolden set columns:")
print(golden_set.columns.tolist())

DF columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']

DF shape:
(18317, 7)

Candidate pool columns:
['customer_tweet_id', 'customer_id', 'customer_created_at', 'customer_text', 'conversation_root', 'historical_replies', 'num_historical_replies', 'text_length']

Training candidates columns:
['customer_tweet_id', 'customer_id', 'customer_created_at', 'customer_text', 'conversation_root', 'historical_replies', 'num_historical_replies', 'text_length', 'candidate_intent', 'sampling_intent']

Training annotations columns:
['customer_tweet_id', 'customer_text', 'conversation_root', 'candidate_intent', 'sampling_intent', 'intent']

Golden set columns:
['customer_tweet_id', 'customer_text', 'conversation_root', 'historical_replies', 'candidate_intent', 'intent', 'should_escalate', 'escalation_reason']


In [ ]:
print("Training annotation label status:")
print(training_annotations["intent"].value_counts(dropna=False))

print("\nTraining candidate sampling intents:")
print(training_annotations["sampling_intent"].value_counts())

print("\nGolden set label status:")
print(golden_set["intent"].value_counts(dropna=False))

Training annotation label status:
intent
    194
Name: count, dtype: int64

Training candidate sampling intents:
sampling_intent
PRODUCT_DEVICE       50
OTHER                50
PRIME                48
ACCOUNT_ACCESS       14
REFUND               12
DELIVERY_DELAY       12
SHIPPING_TRACKING     7
PAYMENT_CHARGE        1
Name: count, dtype: int64

Golden set label status:
intent
    200
Name: count, dtype: int64


In [ ]:
print("First 10 training annotation IDs:")
print(training_annotations[["customer_tweet_id", "sampling_intent", "customer_text"]].head(10).to_string(index=True))

print("\nFirst 10 golden IDs:")
print(golden_set[["customer_tweet_id", "candidate_intent", "intent"]].head(10).to_string(index=True))

First 10 training annotation IDs:
   customer_tweet_id sampling_intent                                                                                                                                                customer_text
0              22579  DELIVERY_DELAY               @AmazonHelp it's 3rd week still I haven't received my replacement order, I had received unworkable mobile &amp; still it's not replace #Amazon
1              21092  DELIVERY_DELAY         @AmazonHelp Order placed 30/10 with stated prime delivery of 31/10. At 7.30pm informed delayed allow further 2 days! Not 1st time prime has failed!!
2                686  DELIVERY_DELAY  @AmazonHelp Already contacted 3-4 times in the last month. But all I got was assurance that matter is escalated &amp; new status will be provided in 1 day!
3              18945  DELIVERY_DELAY                   Disappointed with order# 407-5957878-5349150\nOrdered through prime ID and paid Rs. 50 extra on prime. Still haven't received it.\n@115